# Example 23 — Radiative cooling: a strongly nonlinear ODE

A hot body cooling by radiation to surroundings at $T_\infty$ (lumped capacitance,
Stefan–Boltzmann):
$$\frac{dT}{dt} = -a\,(T^4 - T_\infty^4),\qquad T(0) = T_0 = 2,\ T_\infty = 1.$$
The $T^4$ makes it stiff at the start (cooling rate 15× the linear estimate) and
asymptotically slow later — no elementary closed form, so the reference is RK4.

**PINN design:** hard IC via $T = T_0 + t\,N(t)$; the quartic nonlinearity costs autograd
*nothing* — the same lesson as Burgers' $u u_x$ (Example 10), in one dimension.

Verified: L2 ≈ 5.6e-03 vs RK4, ~3 s on CPU.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

def g1(f, x):
    return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]

A, TINF, T0v, TMAX = 1.0, 1.0, 2.0, 2.0

def rk4_ref(n=4000):
    h = TMAX/n; Tv = T0v; out = [Tv]
    f = lambda T: -A*(T**4 - TINF**4)
    for i in range(n):
        k1=f(Tv); k2=f(Tv+h/2*k1); k3=f(Tv+h/2*k2); k4=f(Tv+h*k3)
        Tv = Tv + h/6*(k1+2*k2+2*k3+k4); out.append(Tv)
    return np.linspace(0, TMAX, n+1), np.array(out)
tr, Tr = rk4_ref()

net = nn.Sequential(nn.Linear(1,32), nn.Tanh(), nn.Linear(32,32), nn.Tanh(),
                    nn.Linear(32,1)).to(device)
T_of = lambda t: T0v + t*net(t)          # hard IC
opt = torch.optim.Adam(net.parameters(), 2e-3)
t0 = time.perf_counter()
for e in range(6000):
    if e == 4000:
        for g in opt.param_groups: g['lr'] = 4e-4
    opt.zero_grad()
    t = (torch.rand(512,1,device=device)*TMAX).requires_grad_(True)
    T = T_of(t)
    res = g1(T,t) + A*(T**4 - TINF**4)
    (res**2).mean().backward(); opt.step()
if device.type=='cuda': torch.cuda.synchronize()
print(f'training: {time.perf_counter()-t0:.0f} s')

tg = torch.tensor(tr, dtype=torch.float32, device=device).reshape(-1,1)
with torch.no_grad(): Tp = T_of(tg).cpu().numpy().ravel()
print(f'L2 vs RK4: {np.sqrt(np.mean((Tp-Tr)**2)):.1e}')

plt.figure(figsize=(8,4))
plt.plot(tr, Tr, 'g', lw=2.4, label='RK4 reference')
plt.plot(tr, Tp, 'r--', lw=1.6, label='PINN')
plt.plot(tr, TINF + (T0v-TINF)*np.exp(-4*A*TINF**3*tr), 'k:', lw=1.2,
         label='linearised (Newton cooling)')
plt.xlabel('t'); plt.ylabel('T'); plt.legend(); plt.grid(alpha=.3)
plt.title(r'Radiative cooling $dT/dt = -a(T^4 - T_\infty^4)$: nonlinearity matters early')
plt.tight_layout(); plt.show()

## Observations
- **The quartic is free.** `T**4` in the residual — no linearisation, no Jacobians. Same
  autograd dividend as Burgers (Ex. 10), minimal setting.
- **Linearised Newton cooling is visibly wrong early** (the dotted curve): at $T=2T_\infty$
  radiation cools ~15× faster than the linear law predicts. A one-plot argument for why
  radiation problems are nonlinear problems.
- **Hard IC** ($T = T_0 + tN$) is the 1-D seed of the trial-function idea used throughout
  Examples 11–20.

**Try:** add convection $-b(T-T_\infty)$ (combined mode); make the emissivity $a$
trainable and recover it from 10 noisy temperature readings (Ex. 22's pattern —
radiometry).